# Prognoosivigade stsenaariumipõhine diagnoos

RQ1 jaoks seotakse vead koormuse, ilma ja kalendri kontekstiga. Lävendid pärinevad ainult treeningandmetest. Stsenaariumid võivad kattuda; tagasivaateline sündmus ei ole prognoosihetkel teadaolev sisend.

In [ ]:
from pathlib import Path
from functools import partial
import sys
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_figure, show_run

RUN = latest_run()
notes = read_json(RUN / "reports/interpretations.json")
table = partial(show_table, RUN, limit=30)
figure = partial(show_figure, RUN)
show_run(RUN)

In [ ]:
thresholds = read_json(RUN / "scenarios/train_thresholds.json")
display(
    Markdown(
        f"Treeningu tipulävend {thresholds['peak']:.2f}, muutuse ülemine lävend {thresholds['increase']:.2f}, alumine {thresholds['decrease']:.2f} ja suure absoluutvea lävend {thresholds['large_error']:.2f} CSV ühikut. Viimane lävend põhineb sobitusjääkidel ning võib üldistamisel probleemide sagedust üle hinnata."
    )
)
table(
    "scenarios/diagnosis.csv",
    columns=[
        "label",
        "n",
        "MAE",
        "RMSE",
        "bias",
        "large_error_share",
        "abs_error_p90",
        "bias_ci_low",
        "bias_ci_high",
    ],
    rename={"bias_ci_low": "Suunatud vea UV alampiir", "bias_ci_high": "Suunatud vea UV ülempiir"},
)
figure("figures/scenarios.png", notes["scenarios"])

In [ ]:
figure("figures/residuals.png", notes["residuals"])
table(
    "scenarios/residual_associations.csv",
    columns=["feature", "outcome", "spearman"],
    rename={"outcome": "Veamõõdiku kood", "spearman": "Spearmani seos"},
)
display(
    Markdown(
        "Tabel kirjeldab suunatud ja absoluutse vea seoseid teadaoleva kontekstiga. Seose suund ja tugevus võivad anda kontrollitava tunnusekatse hüpoteesi, kuid ei tõenda mehhanismi. Aegridade iid-p-väärtusi ei kasutata valiku ega paremuse põhjendusena."
    )
)